# FT-CUR — Ablação de Seleção de Landmarks (4 seletores)

Espelha o estudo feito para o Nyström-SVM, agora no **FT-CUR** (atenção
inter-instâncias comprimida). Compara quatro seletores de *landmarks* mantendo
tudo o mais idêntico:

- **colnorm** — norma de coluna \|x\|² (Drineas–Mahoney), o seletor atual
- **random** — amostragem aleatória uniforme
- **kmeans** — centróides do k-means input-space (Zhang et al. 2008)
- **opposite** — Opposite Maps (kernel k-means feature-space + poda de fronteira, σ por heurística da mediana)

A seleção é one-shot no fit, sobre o input cru (mesmo mecanismo do Nyström-SVM).

**Etapas:** Smoke test → Tier 1 (N≈400) → Tier 2 (N=2000) → Ablação D (N=5000, params fixos).

**Antes de rodar:** Settings → Accelerator → **GPU T4 x2** (ou P100).

**Fluxo pedido:** rode a **Célula 5 (smoke test)** primeiro e cole o resultado
de volta para conferência. Só então libere as células de run completo.

**Resume:** os scripts pulam entradas já completas. Se a sessão cair, baixe os
JSONs em Output, suba como *Dataset* (Add Data) e restaure na Célula 6.


In [ ]:
# Célula 1 — GPU
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0),
          f'| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise RuntimeError('GPU não detectada — Settings > Accelerator > GPU')

In [ ]:
# Célula 2 — Clonar repo e ir para a branch da ablação
import os, subprocess
GIT_URL = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
BRANCH  = 'ftcur-selection-ablation'
PROJECT = '/kaggle/working/sparse-lssvm-transformers-study'
if os.path.exists(PROJECT):
    subprocess.run(['git','-C',PROJECT,'fetch','origin'], check=True)
    subprocess.run(['git','-C',PROJECT,'checkout',BRANCH], check=True)
    subprocess.run(['git','-C',PROJECT,'pull','--rebase','origin',BRANCH], check=True)
else:
    subprocess.run(['git','clone','--branch',BRANCH,GIT_URL,PROJECT], check=True)
os.chdir(PROJECT)
!git log --oneline -2
print('Dir:', os.getcwd())

In [ ]:
# Célula 3 — Dependências
!pip install -q entmax einops scikit-posthocs openpyxl scipy
import torch, sklearn, numpy, scipy
print(f'torch {torch.__version__} | sklearn {sklearn.__version__} | numpy {numpy.__version__} | scipy {scipy.__version__}')

In [ ]:
# Célula 4 — Datasets
!python scripts/download_data.py --tier 1
import sys; sys.path.insert(0, '.')
from src.data.loaders import DatasetLoader
for ds in ['HAB','BCW','TWS','ADULT']:
    X,y,_ = DatasetLoader.load(ds); print(f'  {ds:<8} N={len(y):>6} p={X.shape[1]}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Célula 5 — SMOKE TEST  (rode ISTO primeiro e cole o resultado de volta)
# 4 seletores × HAB × seed 0. ~poucos minutos na T4. Valida que as 4 variantes
# treinam e dão F1 sensato antes de liberar o run completo.
# ═══════════════════════════════════════════════════════════════════════
import json, time
MODELS = ['FTTransformerCURColnorm','FTTransformerCURRandom',
          'FTTransformerCURKmeans','FTTransformerCUROpposite']
t0 = time.time()
!python -u scripts/run_tier1_gridcv.py \
    --models {' '.join(MODELS)} --datasets HAB --seeds 0 \
    --output /kaggle/working/ftcur_smoke.json 2>&1 | tail -6
print(f'\n--- smoke levou {time.time()-t0:.0f}s ---')
d = json.load(open('/kaggle/working/ftcur_smoke.json'))
print(f"{'variante':<26} {'status':<7} {'F1':>7} {'acc':>7} {'m':>4} {'spars':>6} {'fit_s':>6}")
for r in d:
    if r['status']=='ok':
        print(f"{r['variant']:<26} {'ok':<7} {r['test_f1_macro']:7.4f} "
              f"{r['test_accuracy']:7.4f} {r.get('n_support_vectors','?'):>4} "
              f"{r.get('sparsity_ratio',0):6.3f} {r['fit_time_s']:6.0f}")
    else:
        print(f"{r['variant']:<26} ERRO  {r.get('error','')[:70]}")
print('\n>>> COLE ESTA TABELA DE VOLTA NO CHAT PARA CONFERÊNCIA <<<')

In [ ]:
# Célula 6 — Config + resume opcional
import shutil, json
from pathlib import Path
MODELS = ['FTTransformerCURColnorm','FTTransformerCURRandom',
          'FTTransformerCURKmeans','FTTransformerCUROpposite']
TIER1 = ['BCW','PID','HAB','VCP','GCR','AUS','AI4I','TWS','TWM','TWC']
TIER2 = ['ADULT','BANK','CREDIT','HIGGS50K','SHOPPERS','TELCO']
N_SEEDS = 30            # reduza (ex.: 15) se precisar caber na sessão
OUT = Path('results'); OUT.mkdir(exist_ok=True)
F_T1 = 'results/ftcur_sel_tier1.json'
F_T2 = 'results/ftcur_sel_tier2.json'
F_D  = 'results/ftcur_sel_n5000.json'

# --- Resume: se subiu JSONs parciais como dataset, restaure aqui ---
# for name in ['ftcur_sel_tier1.json','ftcur_sel_tier2.json','ftcur_sel_n5000.json']:
#     src = Path(f'/kaggle/input/SEU-DATASET/{name}')
#     if src.exists(): shutil.copy(src, f'results/{name}')

for f in [F_T1,F_T2,F_D]:
    p=Path(f); n=len(json.loads(p.read_text())) if p.exists() else 0
    print(f'{f}: {n} entradas')
print(f'\nPlano: Tier1 {len(MODELS)*len(TIER1)*N_SEEDS} | '
      f'Tier2 {len(MODELS)*len(TIER2)*N_SEEDS} | N5000 {len(MODELS)*len(TIER2)*N_SEEDS} runs')

In [ ]:
# Célula 7 — Tier 1 (4 seletores × 10 datasets × N_SEEDS)   [resumível]
!python -u scripts/run_tier1_gridcv.py \
    --models {' '.join(MODELS)} --datasets {' '.join(TIER1)} \
    --seeds {' '.join(map(str,range(N_SEEDS)))} \
    --output {F_T1} 2>&1 | tee /kaggle/working/run_t1.log | tail -30
import shutil; shutil.copy(F_T1, '/kaggle/working/')
print('Tier 1 salvo em /kaggle/working/')

In [ ]:
# Célula 8 — Tier 2 (4 seletores × 6 datasets × N_SEEDS, N=2000)   [resumível]
!python -u scripts/run_tier2_gridcv.py \
    --models {' '.join(MODELS)} --datasets {' '.join(TIER2)} \
    --seeds {' '.join(map(str,range(N_SEEDS)))} --n-train 2000 \
    --output {F_T2} 2>&1 | tee /kaggle/working/run_t2.log | tail -30
import shutil; shutil.copy(F_T2, '/kaggle/working/')
print('Tier 2 salvo em /kaggle/working/')

In [ ]:
# Célula 9 — Ablação D (N=5000, params FIXOS = moda do GridCV de N=2000)  [heavy]
# Extrai as modas por (variante, dataset) do Tier 2 e injeta no config; roda fixo.
import json, collections
from pathlib import Path
cfg_path = Path('config/tier2_fixed_params.json')
cfg = json.loads(cfg_path.read_text()) if cfg_path.exists() else {}
recs = json.loads(Path(F_T2).read_text())
by = collections.defaultdict(list)
for r in recs:
    if r.get('status')=='ok':
        by[(r['variant'],r['dataset'])].append(tuple(sorted(r['best_params'].items())))
for m in MODELS:
    d = {}
    for ds in TIER2:
        vals = by.get((m,ds))
        if vals:
            mode,_ = collections.Counter(vals).most_common(1)[0]
            d[ds] = dict(mode)
    if d: cfg[m] = d
cfg_path.write_text(json.dumps(cfg, indent=2, sort_keys=True))
print('Modas injetadas para:', [m for m in MODELS if m in cfg])

!python -u scripts/run_tier2_fixedparams.py --n-train 5000 \
    --models {' '.join(MODELS)} --seeds {' '.join(map(str,range(N_SEEDS)))} \
    --output {F_D} 2>&1 | tee /kaggle/working/run_d.log | tail -30
import shutil; shutil.copy(F_D, '/kaggle/working/')
print('Ablação D salva em /kaggle/working/')

In [ ]:
# Célula 10 — Resumo: F1 por seletor + Friedman (nos dados já prontos)
import json, statistics as st
from pathlib import Path
from scipy.stats import friedmanchisquare, wilcoxon
def load(f):
    p=Path(f)
    if not p.exists(): return {}
    out={}
    for r in json.loads(p.read_text()):
        if r.get('status')=='ok': out[(r['variant'],r['dataset'],r['seed'])]=r
    return out
for label,f in [('TIER 1',F_T1),('TIER 2',F_T2),('ABLAÇÃO D N=5000',F_D)]:
    R=load(f)
    if not R: print(f'\n{label}: (sem dados ainda)'); continue
    keyset=lambda m:{(d,s) for (v,d,s) in R if v==m}
    common=set.intersection(*[keyset(m) for m in MODELS]) if all(keyset(m) for m in MODELS) else set()
    print(f'\n=== {label} ({len(common)} pares completos) ===')
    for metric,ml in [('test_f1_macro','F1'),('test_accuracy','acc')]:
        means={m:st.mean(R[(m,d,s)][metric] for (d,s) in common) for m in MODELS}
        line=' | '.join(f"{m.replace('FTTransformerCUR',''):<8} {means[m]:.4f}" for m in MODELS)
        try:
            fp=friedmanchisquare(*[[R[(m,d,s)][metric] for (d,s) in common] for m in MODELS])[1]
        except Exception: fp=float('nan')
        print(f'  {ml:<3}: {line}   [Friedman p={fp:.3f}]')